# TCC: People Analytics — Previsão de Burnout e Turnover em Ambientes de Alta Pressão

---

##  1. Visão Geral e Contexto do Projeto

O aumento da incidência de transtornos de saúde mental no ambiente corporativo — em especial o **Burnout** — tornou-se um dos maiores desafios de gestão de pessoas da era moderna. Em ambientes de alta pressão, como no setor de Tecnologia da Informação e serviços corporativos, a sobrecarga de trabalho, prazos curtos e cultura organizacional disfuncional não apenas impactam a saúde dos colaboradores, mas resultam diretamente em aumento de turnover (rotatividade de pessoal), queda de produtividade e elevados custos operacionais.

Este projeto aplica técnicas de **Data Science e Machine Learning** no contexto de **People Analytics** para identificar os fatores individuais e organizacionais que atuam como preditores da busca por tratamento de saúde mental e do desligamento voluntário de colaboradores.

---

## 2. Problema de Negócio & Objetivos

### **Problema:**
Quais características do trabalhador e do ambiente corporativo predizem a necessidade/busca por tratamento de saúde mental, e qual a relação direta dessas condições com a probabilidade de turnover? O que esses padrões revelam sobre ambientes de trabalho que geram adoecimento?

### **Objetivos Principais:**
1. **Identificação de Preditores:** Mapear os principais fatores de risco (ex: horas extras, histórico de saúde mental, falta de flexibilidade, falta de suporte/benefícios).
2. **Modelagem Preditiva:** Desenvolver modelos supervisionados para prever:
   * A propensão de um colaborador buscar tratamento de saúde mental / apresentar risco de Burnout.
3. **Métricas de Impacto (ROI):** Simular o impacto financeiro do turnover evitado e do investimento em programas preventivos de saúde mental.

---

## 3. Fontes de Dados e Metodologia

Para obter uma visão global do problema, o projeto utiliza base de:

1. **OSMI Mental Health in Tech Survey (Mendeley Data):** Pesquisa focada em saúde mental no setor de tecnologia (visão global/Europa/EUA), abordando percepção de suporte organizacional, tratamento prévio, histórico familiar e estigma no trabalho.

2. **Dados da CAT (Comunicaçao de afastamento do trabalho) / Governo (Dados Oficiais Brasileiros):** Funciona para analisar o fato consumado (o impacto formal). Trata-se de registros oficiais de incapacidade laboral no Brasil, filtrando pelo código de burnout na CID-10 (Classificação internacional das doenças)

#### Este notebook correponde ao tratamento da 2 Base de dados

In [ ]:
## Libs necessárias importadas

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


print('✅ Tudo importado com sucesso!')
print(f'   pandas  {pd.__version__}')
print(f'   numpy   {np.__version__}')
print(f'   seaborn {sns.__version__}')

✅ Tudo importado com sucesso!
   pandas  2.2.2
   numpy   2.0.2
   seaborn 0.13.2


In [ ]:
# Base referente ao mês de Junho de 2023
dados_base = pd.read_csv('/content/D.SDA.PDA.005.CAT.202306.csv', sep=';', encoding='cp1252')


In [ ]:
 # Espiando as colunas da base
dados_base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52831 entries, 0 to 52830
Data columns (total 24 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   Agente  Causador  Acidente   52831 non-null  object
 1   Data Acidente                52831 non-null  object
 2   CBO                          52831 non-null  int64 
 3   CBO.1                        52831 non-null  object
 4   CID-10                       52831 non-null  object
 5   CID-10.1                     52831 non-null  object
 6   CNAE2.0 Empregador           52831 non-null  int64 
 7   CNAE2.0 Empregador.1         52831 non-null  object
 8   Emitente CAT                 52831 non-null  object
 9   Espécie do benefício         52831 non-null  object
 10  Filiação Segurado            52831 non-null  object
 11  Indica Óbito Acidente        52831 non-null  object
 12  Munic Empr                   52831 non-null  object
 13  Natureza da Lesão            52

In [ ]:
dados_base.head(20)

,Agente Causador Acidente,Data Acidente,CBO,CBO.1,CID-10,CID-10.1,CNAE2.0 Empregador,CNAE2.0 Empregador.1,Emitente CAT,Espécie do benefício,...,Origem de Cadastramento CAT,Parte Corpo Atingida,Sexo,Tipo do Acidente,UF Munic. Acidente,UF Munic. Empregador,Data Afastamento,Data Acidente.1,Data Nascimento,Data Acidente.2
0,Temperatura Muito Ba,31/05/2023,841815,841815-Oper. Máquina,T139,T13.9 Traum Ne do Me,1093,Fabricacao de Produt,Empregador,Pa,...,Internet,Perna (Entre O Torno,Masculino,Típico,Maranhão,São Paulo,00/00/0000,31/05/2023,26/06/1987,31/05/2023
1,Atrito ou Abrasao po,30/05/2023,514205,514205-Coletor Lixo,T159,T15.9 Corpo Estranho,3811,Coleta de Residuos N,Empregador,Pa,...,Internet,Olho (Inclusive Nerv,Masculino,Típico,{ñ class},Amazonas,00/00/0000,30/05/2023,20/06/2001,30/05/2023
2,Temperatura Muito Al,31/05/2023,762005,762005-Trab. Polival,T240,T24.0 Queim Quadr Me,1510,Curtimento e Outras,Empregador,Pa,...,Internet,Quadris (Inclusive P,Masculino,Típico,{ñ class},Rio Grande do Sul,00/00/0000,31/05/2023,01/08/1983,31/05/2023
3,"Atrito ou Abrasao, N",30/05/2023,313315,313315-Tec. de Telec,S300,S30.0 Contusao do Do,6311,"Tratamento de Dados,",Empregador,Pa,...,Internet,Braco (Entre O Punho,Masculino,Típico,{ñ class},Amazonas,00/00/0000,30/05/2023,31/12/1992,30/05/2023
4,Rua e Estrada - Supe,30/05/2023,422315,422315-Oper. Telemar,V288,{ñ class},1411,Confeccao de Roupas,Empregador,Pa,...,Internet,Partes Multiplas - A,Feminino,Trajeto,{ñ class},Santa Catarina,00/00/0000,30/05/2023,07/12/1992,30/05/2023
5,{ñ class},31/05/2023,324205,324205-Tec. em Patol,S499,S49.9 Traum Ne do Om,8412,Regulacao das Ativid,Segurado/Dependente,Pa,...,Internet,Dedo,Feminino,Típico,{ñ class},Bahia,00/00/0000,31/05/2023,03/04/1975,31/05/2023
6,Esforco Excessivo ao,29/05/2023,322205,322205-Tec. de Enfer,Z209,Z20.9 Contato Exposi,8610,Atividades de Atendi,Empregador,Pa,...,Internet,Olho (Inclusive Nerv,Feminino,Típico,Piauí,Sergipe,00/00/0000,29/05/2023,25/01/1984,29/05/2023
7,{ñ class},30/05/2023,322205,322205-Tec. de Enfer,Z042,Z04.2 Exame e Observ,8610,Atividades de Atendi,Empregador,Pa,...,Internet,Dedo,Feminino,Típico,Tocantins,Rio de Janeiro,00/00/0000,30/05/2023,22/06/1987,30/05/2023
8,Impacto Sofrido por,31/05/2023,782220,782220-Oper. de Empi,S600,S60.0 Contusao de De,5240,Atividades Auxiliare,Empregador,Pa,...,Internet,Dedo,Masculino,Típico,Maranhão,São Paulo,00/00/0000,31/05/2023,20/06/1987,31/05/2023
9,Metal - Inclui Liga,31/05/2023,783210,783210-Carregador (A,S610,S61.0 Ferim de Dedos,4622,Comercio Atacadista,Empregador,Pa,...,Internet,Mao (Exceto Punho ou,Masculino,Típico,{ñ class},Rio Grande do Sul,00/00/0000,31/05/2023,17/09/2003,31/05/2023


In [ ]:
dados_base.shape

(52831, 24)

As bases de dados estão separados por mês são aproximadamente 36 meses separados base por base, então faremos a concatenação (união vertical delas) de todas para que os meses fiquem consolidados em uma base só.

In [ ]:
import glob #<- - Busca nos caminhos de arquivos no computador usando padrões de texto (wildcards/Asteriscos). Evitando que digitemos arquivo por arquivo.
# Todos os arquivos possuem o mesmo nome, mudando apenas o mês e ano então isso facilita esse trabalho.

In [ ]:
arquivo_cat = glob.glob("D.SDA.PDA.005.CAT.*")

dfs = []
for arquivo in arquivo_cat:
  df_mes = pd.read_csv(arquivo, sep=';', encoding='cp1252')
  dfs.append(df_mes)

## Juntando todos os meses em uma única base

df_cat_completo = pd.concat(dfs, ignore_index=True)


print(
    f"Base consolidada com sucesso! Total de linhas: {len(df_cat_completo):,}"
)

Base consolidada com sucesso! Total de linhas: 1,739,703


A base ficou com mais de 1 milhão de linhas, mas como vamos utilizar a estretégia pra filtrar somentes os códigos correpondente à saúde mental, reduziremos bastante a nossa base.

- Os códigos da CID-10 filtrados serão:

- Z73.0: Esgotamento (síndrome de burnout) ou sensação de estar acabado.
- F41: Outros transtornos ansiosos (como ansiedade generalizada ou pânico).
- F32: Episódio depressivo único (leve, moderado ou grave).
- F33: Transtorno depressivo recorrente (quando a pessoa tem vários episódios de depressão ao longo da vida).
- F43: Reações ao estresse grave e transtornos de adaptação (como o estresse pós-traumático).

In [ ]:

cids_saude_mental = ["Z73.0", "Z730", "F41", "F32", "F33", "F43"]
df_burnout = df_cat_completo[df_cat_completo["CID-10"].str.contains('|'.join(cids_saude_mental), na=False)]

print(
    f"Base Filtrada com sucesso! Total de linhas: {len(df_burnout):,}"
)

Base Filtrada com sucesso! Total de linhas: 18,717


In [ ]:
df_burnout.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18717 entries, 202 to 1739632
Data columns (total 28 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Agente  Causador  Acidente   18717 non-null  object 
 1   Data Acidente                18717 non-null  object 
 2   CBO                          18717 non-null  int64  
 3   CBO.1                        18717 non-null  object 
 4   CID-10                       18717 non-null  object 
 5   CID-10.1                     18717 non-null  object 
 6   CNAE2.0 Empregador           18717 non-null  int64  
 7   CNAE2.0 Empregador.1         18717 non-null  object 
 8   Emitente CAT                 18717 non-null  object 
 9   Espécie do benefício         18717 non-null  object 
 10  Filiação Segurado            18717 non-null  object 
 11  Indica Óbito Acidente        18717 non-null  object 
 12  Munic Empr                   18717 non-null  object 
 13  Natureza da Lesão

In [ ]:
df_burnout.head()

,Agente Causador Acidente,Data Acidente,CBO,CBO.1,CID-10,CID-10.1,CNAE2.0 Empregador,CNAE2.0 Empregador.1,Emitente CAT,Espécie do benefício,...,UF Munic. Acidente,UF Munic. Empregador,Data Afastamento,Data Despacho Benefício,Data Acidente.1,Data Nascimento,Data Emissão CAT,Tipo de Empregador,CNPJ/CEI Empregador,Data Acidente.2
202,Agente do Acidente I,20/03/2025,373140,{ñ class},F332NU,{ñ class},4742,Comercio Varejista d,Empregador,Pa,...,Pará,Pernambuco,20/03/2025,00/00/0000,20/03/2025,12/08/1998,23/07/2025,Cnpj/Cgc,3.756359e+13,NaN
423,{ñ class},22/07/2025,519110,519110-Motociclista,F411NU,{ñ class},4530,Comercio de Pecas e,Empregador,Pa,...,Maranhão,São Paulo,15/07/2025,00/00/0000,22/07/2025,12/12/1992,24/07/2025,Cnpj/Cgc,1.107753e+13,NaN
596,"Ataque de Ser Vivo,",28/05/2025,141705,141705-Ger. Prod. Ba,Z730NU,{ñ class},6422,"Bancos Multiplos, co",Sindicato,Pa,...,Maranhão,São Paulo,28/05/2025,00/00/0000,28/05/2025,08/01/1976,10/07/2025,Cnpj/Cgc,9.040089e+13,NaN
702,{ñ class},12/06/2025,232105,232105-Prof. Artes n,F412NU,{ñ class},8541,Educacao Profissiona,Segurado/Dependente,Pa,...,Tocantins,Rio de Janeiro,12/06/2025,00/00/0000,12/06/2025,11/11/1982,16/07/2025,Cnpj/Cgc,3.160876e+13,NaN
713,"Agente do Acidente,",02/05/2025,252210,252210-Contador,F321NU,{ñ class},6110,Telecomunicacoes por,Sindicato,Pa,...,Maranhão,São Paulo,02/05/2025,00/00/0000,02/05/2025,28/06/1978,31/07/2025,Cnpj/Cgc,2.558157e+12,NaN


- Temos 3 colunas de nome "data do acidente", vamos verificar se as duas primeirad são iguais, já que a terceira contém vários nulos e provavelmente não utilizaremos ela.

In [ ]:
diferencas = (
    df_burnout['Data Acidente'] != df_burnout['Data Acidente.1']
).sum()
print(f'Quantidade de linhas onde Data Acidente é diferente da .1: {diferencas}')

Quantidade de linhas onde Data Acidente é diferente da .1: 0


Como elas são iguais vamos manter somente a primeira.

A próxima investigação será sobre entender se a coluna "Data Afastamento" possui data zerada em todas as linhas da coluna, se sim vamos exclui-la também

In [ ]:
print(df_burnout['Data  Afastamento'].value_counts(dropna=False).head(10))

Data  Afastamento
00/00/0000    3344
04/08/2025     100
01/07/2025      92
02/07/2025      91
30/07/2025      87
02/06/2025      78
14/07/2025      78
11/07/2025      77
28/07/2025      77
12/08/2025      77
Name: count, dtype: int64


### 1. Primeiras estratégias para começar a Limpeza de Dados

####1.1 Os primeiros passos

- Renomear os nomes das colunas pois como exemplo "Data Afastamento" possui mais de um espaço

- Excluir a coluna sensível de CNPJ/CEI Empregador

- Remover 2 das 3 colunas de Data do acidente

- Alterar as Datas para seu tipo correto pois elas estão como: "Object" e devem ser "Datetime". Dessa forma também conseguimos fazer calculos com as datas para saber por exemplo a idade do colaborador no momento do registro de afastamento.

In [ ]:
colunas_mantidas = [
'Agente  Causador  Acidente',
'Data Acidente',
'CBO',
'CBO.1',
'CID-10',
'CID-10.1',
'CNAE2.0 Empregador',
'CNAE2.0 Empregador.1',
'Emitente CAT',
'Espécie do benefício',
'Filiação Segurado',
'Indica Óbito Acidente',
'Munic Empr',
'Natureza da Lesão',
'Origem de Cadastramento CAT',
'Parte Corpo Atingida',
'Sexo',
'Tipo do Acidente',
'UF  Munic.  Acidente',
'UF Munic. Empregador',
'Data  Afastamento',
'Data Despacho Benefício',
'Data Nascimento',
'Data Emissão CAT',
'Tipo de Empregador'
]

# Filtrando e criando uma copia, garantindo que as colunas desnecessárias não acompanharão a cópia
df_burnout_filtrado = df_burnout[colunas_mantidas].copy()


- Renomear as colunas

In [ ]:
# Dicionário

renomear_colunas = {
'Agente  Causador  Acidente': 'agente_causador',
'Data Acidente': 'data_acidente',
'CBO': 'cbo_codigo',
'CBO.1': 'cbo_descricao',
'CID-10': 'cid_codigo',
'CID-10.1': 'cid_descricao',
'CNAE2.0 Empregador': 'cnae_codigo',
'CNAE2.0 Empregador.1': 'cnae_descricao',
'Emitente CAT': 'emitente_cat',
'Espécie do benefício': 'especie_beneficio',
'Filiação Segurado': 'filiacao_segurado',
'Indica Óbito Acidente': 'indica_obito',
'Munic Empr': 'municipio_empregador',
'Natureza da Lesão': 'natureza_lesao',
'Origem de Cadastramento CAT': 'origem_cadastramento',
'Parte Corpo Atingida': 'parte_corpo_atingida',
'Sexo': 'genero',
'Tipo do Acidente': 'tipo_acidente',
'UF  Munic.  Acidente': 'uf_acidente',
'UF Munic. Empregador': 'uf_empregador',
'Data  Afastamento': 'data_afastamento',
'Data Despacho Benefício': 'data_despacho_beneficio',
'Data Nascimento': 'data_nascimento',
'Data Emissão CAT': 'data_emissao_cat',
'Tipo de Empregador': 'tipo_empregador'
}


## Aplicando os renomes

df_burnout_filtrado.rename(columns=renomear_colunas, inplace=True)

In [ ]:
# Convertendo as datas para o formato date time e calcular a idade do trabalhador

df_burnout_filtrado['data_nascimento_dt'] = pd.to_datetime(
    df_burnout_filtrado['data_nascimento'], format='%d/%m/%Y', errors='coerce'
)

df_burnout_filtrado['data_acidente_dt'] = pd.to_datetime(
    df_burnout_filtrado['data_acidente'], format='%d/%m/%Y', errors='coerce'
)


## Calcular a idade em anos no momento do registro

df_burnout_filtrado['idade'] = (
    (df_burnout_filtrado['data_acidente_dt'] - df_burnout_filtrado['data_nascimento_dt']).dt.days / 365.25
).round(0)


# Limpando as colunas auxiliares
df_burnout_filtrado.drop(columns=['data_nascimento_dt', 'data_acidente_dt'], inplace=True)

Alterando todas as colunas de datas para o seu tipo correspondente: DateTime

In [ ]:
colunas_datas = [
    'data_acidente',
    'data_afastamento',
    'data_despacho_beneficio',
    'data_emissao_cat',
    'data_nascimento'
]

# Convertendo todas automaticamente

for col in colunas_datas:
  df_burnout_filtrado[col] = pd.to_datetime(df_burnout_filtrado[col], format='%d/%m/%Y', errors='coerce')

In [ ]:
df_burnout_filtrado.to_csv(
    'cat_saude_mental_brasil_tratado.csv', index=False, sep=';', encoding='utf-8'
)

print(' Conversão de datas concluída com sucesso!')
print(df_burnout_filtrado[colunas_datas + ['idade']].dtypes)

 Conversão de datas concluída com sucesso!
data_acidente              datetime64[ns]
data_afastamento           datetime64[ns]
data_despacho_beneficio    datetime64[ns]
data_emissao_cat           datetime64[ns]
data_nascimento            datetime64[ns]
idade                             float64
dtype: object


- Análisando como estão os dados sobre gênero para verificar se precisamos fazer a limpeza nela já será uma coluna extremamente relevante para as nossas análises.





In [ ]:
# Análise descritiva preliminar

print("###### Distribuiçao por gênero #######")
qtd_genero = df_burnout_filtrado['genero'].value_counts(dropna=False)
porcentagem_genero = df_burnout_filtrado['genero'].value_counts(
    normalize=True, dropna=False
) * 100

df_genero = pd.DataFrame({'Total Casos': qtd_genero, 'Percentual (%)': porcentagem_genero})
print(df_genero.round(2))


print('\n####### estatisticas da idade #######\n')
print(f"Idade Média: {df_burnout_filtrado['idade'].mean():.1f} anos")
print(f"Mediana da Idade: {df_burnout_filtrado['idade'].median():.1f} anos")
print(f"Idade Mínima: {df_burnout_filtrado['idade'].min():.0f} anos")
print(f"Idade Máxima: {df_burnout_filtrado['idade'].max():.0f} anos")

###### Distribuiçao por gênero #######
                      Total Casos  Percentual (%)
genero                                           
Feminino                    11665           62.32
Masculino                    7041           37.62
Não Informado                   9            0.05
Indeterminado                   2            0.01

####### estatisticas da idade #######

Idade Média: 39.7 anos
Mediana da Idade: 39.0 anos
Idade Mínima: 16 anos
Idade Máxima: 74 anos


Temos como resultado 2 categorias a mais na coluna de gênero:
- Não Informado = 9 Registros
- Indeterminado = 2 Registros

Como representa muito pouco sobre a base inteira vamos excluir esses registros juntamente com as duas categorias, deixando apenas gênero: Masculino e Feminino.
Na CAT, colunas como **CBO (profissão), CID** ou **Estado** não possuem relação determinística com gênero que garanta a dedução sem inventar/armazenar viés.

In [ ]:
# Limpamos os nomes das colunas, agora vamos olhar com mais carinho para os registros das colunas, limpando espaços em branco ou valores inconsistentes que possam mais a frente quebrar a nossa análise.

# Removendo espaços em branco nas extremidades dos registros das colunas de texto
colunas_texto = df_burnout_filtrado.select_dtypes(include=['object']).columns

for col in colunas_texto:
  df_burnout_filtrado[col] = df_burnout_filtrado[col].astype(str).str.strip()

# Mapeando apenas os generos Femininos e Masculinos, transformando automaticamente os outros em nulos
mapeamento_genero = {
    'Feminino': 'Feminino',
    'Masculino': 'Masculino',
}

df_burnout_filtrado['genero'] = df_burnout_filtrado['genero'].map(mapeamento_genero)


## Removendo os registros nao identificados juntamente com a mesma categoria
linhas_antes = len(df_burnout_filtrado)
df_burnout_filtrado = df_burnout_filtrado.dropna(subset=['genero']).copy()
linhas_depois = len(df_burnout_filtrado)

# Validando o Resultado

print(f'Limpeza concluida: {linhas_antes - linhas_depois} registros sem gênero'
' definido foram removidos.')

print(f'Total de registros válidos na base: {linhas_depois:,}')


## Validando a nova distribuiçao após a limpeza

print('\n###### distribuiçao de gênero pós Limpeza ')
print(df_burnout_filtrado['genero'].value_counts(dropna=False))


Limpeza concluida: 11 registros sem gênero definido foram removidos.
Total de registros válidos na base: 18,706

###### distribuiçao de gênero pós Limpeza 
genero
Feminino     11665
Masculino     7041
Name: count, dtype: int64


- Olhando um pouco mais para as inconsitências de outras colunas da base

In [ ]:
checar_colunas = [
    'cid_codigo',
    'uf_acidente',
    'emitente_cat',
    'tipo_acidente',
]

for col in checar_colunas:
  if col in df_burnout_filtrado.columns:
    print(
        df_burnout_filtrado[col].value_counts().head(5)
    )
    print('-' * 40)

cid_codigo
Z730    5936
F411    2772
F412    1605
F431    1322
F430    1121
Name: count, dtype: int64
----------------------------------------
uf_acidente
Maranhão     7792
{ñ class}    4007
Tocantins    2520
Rondônia     1462
Ceará         963
Name: count, dtype: int64
----------------------------------------
emitente_cat
Sindicato              6916
Empregador             5419
Segurado/Dependente    4629
Médico                  878
Autoridade Pública      826
Name: count, dtype: int64
----------------------------------------
tipo_acidente
Doença     14448
Típico      3547
Trajeto      711
Name: count, dtype: int64
----------------------------------------


Com essa verificaçao conseguimos identificar que existe uma quantidade considerável de dados em UF como: {ñ class} e UF geralmente é apenas a sigla do estado nesse caso está descrito por extenso.
Só aparecem alguns estados porque a coluna reflete exatamente o municipio em que o acidente ocorreu.
Aqui faremos um cruzamento também com a coluna `uf_empregador`(que trata da sede da empresa contratante) para entender qual está mais completa e como prosseguiremos com a limpeza

In [ ]:
print("#### Comparaçao entre as colunas: UF_ACIDENTE vs UF_EMPREGADOR ####")

print(df_burnout_filtrado[['uf_acidente', 'uf_empregador']].value_counts().head())


## Se a uf_acidente for {ñ class} podemos preencher com a uf_empregador?

uf_recuperaveis = df_burnout_filtrado[df_burnout_filtrado['uf_acidente'] == '{ñ class}']['uf_empregador'].value_counts()
print('\n ###### UFs de Empregador para os casos não classificados #######')
print(uf_recuperaveis)

#### Comparaçao entre as colunas: UF_ACIDENTE vs UF_EMPREGADOR ####
uf_acidente  uf_empregador    
Maranhão     São Paulo            7759
Tocantins    Rio de Janeiro       2492
Rondônia     Minas Gerais         1433
Ceará        Distrito Federal      956
{ñ class}    Rio Grande do Sul     948
Name: count, dtype: int64

 ###### UFs de Empregador para os casos não classificados #######
uf_empregador
Rio Grande do Sul      948
Bahia                  733
Santa Catarina         528
Ceará                  403
Rio Grande do Norte    233
Espírito Santo         216
Goiás                  176
Alagoas                164
Maranhão               140
Mato Grosso do Sul     139
Amazonas               139
Mato Grosso            131
São Paulo               24
Zerado                   8
Minas Gerais             8
Distrito Federal         7
Pernambuco               5
Rio de Janeiro           2
Paraíba                  1
Paraná                   1
Acre                     1
Name: count, dtype: int64


- Temos então um padrão: onde o adoecimento não ocorre em um local físico pontual como uma "obra", ou escritório em outra cidade, por exemplo, mas sim no contexto do trabalho em si o preenchimento ficou como {ñ class}.

Além disso foi encontrado registro como: Zerado entre as UF.

E UF está como extenso em todos os registros quando na verdade UF é somente o código do estado.

Vamos tratar desse problema por partes:

1. Substituir {ñ class} por NaN na `uf_incidente`.

2. Preencher os valores nulos de `uf_acidente` com o valor da `uf_empregador`

3. Mapear os nomes por extenso e trocar pela sigla oficial da UF

4. Tratar o valor `Zerado` transformado-o em nulo e descartando


In [ ]:
## Tratando as ufs nao classificadas
sujeiras = ['{ñ class}', 'Zerado', 'None', '']
df_burnout_filtrado['uf_acidente'] = df_burnout_filtrado['uf_acidente'].replace(sujeiras, np.nan)

df_burnout_filtrado['uf_empregador'] = df_burnout_filtrado['uf_empregador'].replace(sujeiras, np.nan)


## Onde uf_acidente for nula, assumirá uf_empregador

df_burnout_filtrado['uf_consolidada'] = df_burnout_filtrado['uf_acidente'].fillna(df_burnout_filtrado['uf_empregador'])


## Padronizando e mapeando as UF's

mapa_ufs = {
    'Acre': 'AC',
    'Alagoas': 'AL',
    'Amapá': 'AP',
    'Amazonas': 'AM',
    'Bahia': 'BA',
    'Ceará': 'CE',
    'Distrito Federal': 'DF',
    'Espírito Santo': 'ES',
    'Goiás': 'GO',
    'Maranhão': 'MA',
    'Mato Grosso': 'MT',
    'Mato Grosso do Sul': 'MS',
    'Minas Gerais': 'MG',
    'Pará': 'PA',
    'Paraíba': 'PB',
    'Paraná': 'PR',
    'Pernambuco': 'PE',
    'Piauí': 'PI',
    'Rio de Janeiro': 'RJ',
    'Rio Grande do Norte': 'RN',
    'Rio Grande do Sul': 'RS',
    'Rondônia': 'RO',
    'Roraima': 'RR',
    'Santa Catarina': 'SC',
    'São Paulo': 'SP',
    'Sergipe': 'SE',
    'Tocantins': 'TO',
}

### Coluna oficial com as siglas para os posteriores gráficos
df_burnout_filtrado['uf_final'] = df_burnout_filtrado['uf_consolidada'].map(mapa_ufs)


### Validando o resultado

print('###### Top 10 estados após resgate e padronização ######')
print(df_burnout_filtrado['uf_final'].value_counts(dropna=False).head(15))

print(
    '\nRegistros sem UF após o resgate: '
    f'{df_burnout_filtrado['uf_final'].isna().sum()}'
)

###### Top 10 estados após resgate e padronização ######
uf_final
MA    7932
TO    2520
RO    1462
CE    1366
RS     948
BA     733
PA     623
SC     528
RR     483
AM     397
RN     233
AC     220
ES     216
AL     176
GO     176
Name: count, dtype: int64

Registros sem UF após o resgate: 65


Foi criado uma coluna auxiliar `uf_consolidada` que copia a coluna `uf_empregador` olha para a coluna `uf_acidente` e onde está nulo (que era {ñ class}) ela preenche com o registro que está em `uf_empregador` que é a sede da empresa contratante.

Após isso foi criado outra coluna que reflete a `uf_acidente` já preenchida onde estava {ñ class} e transforma os nomes dos estados por extenso em siglas da ufs.

- A coluna `uf_final` será renomeada para `uf_acidente` pois ela está preenchida com as siglas corretas dos estados. e removeremos a `uf_acidente` original por possuir muitos valores nulos.

- Manteremos a coluna `uf_empregador` pois ela traz a localização real da sede da empresa e o vinculo do trabalhador, além de abranger uma variedade maior dos estados do Brasil, o que pode ser ótimo para os gráficos futuros.

- Na análise abaixo podemos ver que a coluna `data_despacho_beneficio` ficou totalmente nula por algum erro de digitação ou por conter datas zeradas, o Python não consegue identificar mes e ano zerados porque nao existe, então decidimos excluir essa coluna pois nesse contexto da análise ela também não será muito relevante.

- A coluna auxiliar `uf_consolidada` também será removida porque ela só serviu de apoio para o objetivo do processo de limpeza.

In [ ]:
df_burnout_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18706 entries, 202 to 1739632
Data columns (total 28 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   agente_causador          18706 non-null  object        
 1   data_acidente            18706 non-null  datetime64[ns]
 2   cbo_codigo               18706 non-null  int64         
 3   cbo_descricao            18706 non-null  object        
 4   cid_codigo               18706 non-null  object        
 5   cid_descricao            18706 non-null  object        
 6   cnae_codigo              18706 non-null  int64         
 7   cnae_descricao           18706 non-null  object        
 8   emitente_cat             18706 non-null  object        
 9   especie_beneficio        18706 non-null  object        
 10  filiacao_segurado        18706 non-null  object        
 11  indica_obito             18706 non-null  object        
 12  municipio_empregador     18706 no

In [ ]:
### Removendo a coluna auxiliar e a data zerada

colunas_remover = [
    'uf_acidente', #<-- remove a coluna original com valores nulos
    'uf_consolidada', #<-- remove a coluna auxiliar
    'data_despacho_beneficio', #<-- Remove a coluna que ficou 100% NaT
]

df_burnout_filtrado.drop(
    columns=[
        col
        for col in colunas_remover
        if col in df_burnout_filtrado.columns], inplace=True,
)

## Renomeando uf_final(COM AS SIGLAS) para uf_acidente
df_burnout_filtrado.rename(columns={'uf_final': 'uf_acidente'}, inplace=True)


## Validadando o Resultado

print('##### Colunas de UF preservadas #####')
print(
    df_burnout_filtrado[['uf_acidente', 'uf_empregador']].value_counts(dropna=False).head(10)
)

##### Colunas de UF preservadas #####
uf_acidente  uf_empregador    
MA           São Paulo            7759
TO           Rio de Janeiro       2492
RO           Minas Gerais         1433
CE           Distrito Federal      956
RS           Rio Grande do Sul     948
BA           Bahia                 733
PA           Pernambuco            616
SC           Santa Catarina        528
RR           Paraná                468
CE           Ceará                 403
Name: count, dtype: int64


In [ ]:
df_burnout_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18706 entries, 202 to 1739632
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   agente_causador       18706 non-null  object        
 1   data_acidente         18706 non-null  datetime64[ns]
 2   cbo_codigo            18706 non-null  int64         
 3   cbo_descricao         18706 non-null  object        
 4   cid_codigo            18706 non-null  object        
 5   cid_descricao         18706 non-null  object        
 6   cnae_codigo           18706 non-null  int64         
 7   cnae_descricao        18706 non-null  object        
 8   emitente_cat          18706 non-null  object        
 9   especie_beneficio     18706 non-null  object        
 10  filiacao_segurado     18706 non-null  object        
 11  indica_obito          18706 non-null  object        
 12  municipio_empregador  18706 non-null  object        
 13  natureza_lesao   

### Olhando um pouco mais minuciosamente para as colunas individualmente e entendendo como elas estão representadas, se permaneceremos com elas ou se a exige tratamento

In [ ]:
## Olhando como está a coluna de municipio do empregador

df_burnout_filtrado['municipio_empregador'].value_counts()

,count
municipio_empregador,
355030-São Paulo,4914
330455-Rio de Janeir,1010
530010-Brasília,982
353440-Osasco,717
431490-Porto Alegre,530
...,...
150320-Igarapé-Açu,1
352610-Juquiá,1
150340-Inhangapi,1


- Aqui existe um clássico: a mistura entre número e texto, essa coluna está representada como texto porém possui números. Como nas visualizações é mais fácil identificar o nome do municipio do que o código dele, é interessante padronizarmos essa coluna

In [ ]:
## Olhando como está a coluna de origem_cadastramento

df_burnout_filtrado['origem_cadastramento'].value_counts()

,count
origem_cadastramento,
Internet,10470
{ñ class},8236


Aqui temos duas classes de cadastramento
- Internet
- Não Identificada

Como {ñ class} trata-se de um número consideravel da base (~44%) também vamos precisar trata-la.
Considerando que o contrário do cadastramento por internet seria presencial, trocaremos {ñ class} por Presencial

In [ ]:
## Olhando como está a coluna de parte_corpo_atingida

df_burnout_filtrado['parte_corpo_atingida'].value_counts()


,count
parte_corpo_atingida,
Sistema Nervoso,12246
"Cabeca, Nic",2588
Localizacao da Lesao,1390
Sistemas e Aparelhos,951
"Cabeca, Partes Multi",434
Cranio (Inclusive En,255
Partes Multiplas - A,166
"Face, Partes Multipl",106
"Membros Superiores,",104


`'parte_corpo_atingida'`: Sistema Nervoso representa a grande maioria dos casos (12.246), seguido por Cabeça, Nic (2.588) e Sistemas e Aparelhos (951). Isso ratifica a natureza psíquica/neurológica do Burnout na base do INSS e contrasta diretamente com lesões físicas tradicionais (ex: Membros Superiores ou Mão).

In [ ]:
## Olhando como está a coluna de especie_beneficio

df_burnout_filtrado['especie_beneficio'].value_counts()

,count
especie_beneficio,
Pa,18703
Auxílio Doenca por A,3


`'especie_beneficio'`: Essa podemos deletar por se tratar de um único tipo de beneficio Pa, e 3 casos isolados de auxilio doença por A. Essa coluna não possui variância. Vejo que ela não é relevante para o contexto da nossa análise.

In [ ]:
## Olhando como está a coluna de filiacao_segurado

df_burnout_filtrado['filiacao_segurado'].value_counts()

,count
filiacao_segurado,
Empregado,18647
Trabalhador Avulso,34
{ñ class},19
Segurado Especial,6


`'filiacao_segurado'`: Também possui uma variância baixa. Acredito que ela também não será tão interessante e remove-la ajuda a enxugar o modelo

In [ ]:
### Código para o tratamento das colunas acima

# Dropando as colunas que não serão necessárias para a análise
colunas_sem_variancia = ['especie_beneficio', 'filiacao_segurado']
df_burnout_filtrado.drop(
    columns= [
        col
        for col in colunas_sem_variancia
        if col in df_burnout_filtrado.columns
    ], inplace=True,
)


## tratando a origem de cadastramento
df_burnout_filtrado['origem_cadastramento'] = (
    df_burnout_filtrado['origem_cadastramento'].replace(['{ñ class}'], 'Presencial').fillna('Presencial')
)


#Extraindo apenas o nome do munícipio e descartando o código do IBGE
df_burnout_filtrado['municipio_empregador'] = (
    df_burnout_filtrado['municipio_empregador']
    .astype(str)
    .str.split('-', n=1)
    .str[1]
    .str.strip()
)


## Tratando possíveis erros ou nulos caso o municipio venha sem hífen
df_burnout_filtrado['municipio_empregador'] = (
    df_burnout_filtrado['municipio_empregador']
    .replace(['', 'nan', '{\ñ class}', '{ñ class}'], np.nan)
    .fillna('Não informado')
)


# Tratando a 'parte_corpo_atingida'
df_burnout_filtrado['parte_corpo_atingida'] = (
    df_burnout_filtrado['parte_corpo_atingida']
    .replace(['{\ñ class}', '{ñ class}'], 'Não Informado')
    .fillna('Não Informado')
)


print('\n####### Origem de cadastramento cadastrado #######\n')
print(df_burnout_filtrado['origem_cadastramento'].value_counts())

print('\n##### 5 principais partes do corpo atingida #######\n')
print(df_burnout_filtrado['parte_corpo_atingida'].value_counts().head())

print('\n###### top 10 municipios empregador (somente nomes) #######\n')
print(df_burnout_filtrado['municipio_empregador'].value_counts().head(10))


####### Origem de cadastramento cadastrado #######

origem_cadastramento
Internet      10470
Presencial     8236
Name: count, dtype: int64

##### 5 principais partes do corpo atingida #######

parte_corpo_atingida
Sistema Nervoso         12246
Cabeca, Nic              2588
Localizacao da Lesao     1390
Sistemas e Aparelhos      951
Cabeca, Partes Multi      434
Name: count, dtype: int64

###### top 10 municipios empregador (somente nomes) #######

municipio_empregador
São Paulo        4914
Rio de Janeir    1010
Brasília          982
Osasco            717
Porto Alegre      531
Belo Horizont     355
Juiz de Fora      335
Salvador          316
Recife            295
Fortaleza         248
Name: count, dtype: int64


In [ ]:
df_burnout_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18706 entries, 202 to 1739632
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   agente_causador       18706 non-null  object        
 1   data_acidente         18706 non-null  datetime64[ns]
 2   cbo_codigo            18706 non-null  int64         
 3   cbo_descricao         18706 non-null  object        
 4   cid_codigo            18706 non-null  object        
 5   cid_descricao         18706 non-null  object        
 6   cnae_codigo           18706 non-null  int64         
 7   cnae_descricao        18706 non-null  object        
 8   emitente_cat          18706 non-null  object        
 9   indica_obito          18706 non-null  object        
 10  municipio_empregador  18706 non-null  object        
 11  natureza_lesao        18706 non-null  object        
 12  origem_cadastramento  18706 non-null  object        
 13  parte_corpo_ating

In [ ]:
df_burnout_filtrado['uf_acidente'].isna().sum()

np.int64(65)

In [ ]:
df_burnout_filtrado['data_emissao_cat'].isna().sum()

np.int64(2530)

In [ ]:
df_burnout_filtrado['uf_empregador'].isna().sum()

np.int64(73)

In [ ]:
df_burnout_filtrado['data_afastamento'].isna().sum()

np.int64(3353)

Ainda temos nulos em algumas colunas importantes em uf_acidente e uf_empregador. Aqui o ideal é cruzar novamente essas duas e entender se uma possui a informaçào que a outra precisa. O que continuar nulo nas duas preenchemos depois com 'não informado'.

Assim como as colunas de datas: `data_emissao_cat` e `data_afastamento`, essas duas possuem uma quantidade relevante de nulos e devem ser olhadas com atenção. Podemos usar a data acidente que está preenchida para resolver o caso da colunas `data_emissao_cat`, pois na imensa maioria dos registros do INSS, a data de emissão é idêntica ou muito próxima à `data_acidente`.

Porém nem todo caso registrado na CAT gera afastamento imediato, se inferirmos a data do acidente na coluna de `data_afastamento`, podemos inventar e distorcer o indicador real de afastamento.


In [ ]:
## Resgate mútuo entre as UF`s

df_burnout_filtrado['uf_acidente'] = df_burnout_filtrado['uf_acidente'].fillna(df_burnout_filtrado['uf_empregador'])

df_burnout_filtrado['uf_empregador'] = df_burnout_filtrado['uf_empregador'].fillna(df_burnout_filtrado['uf_acidente'])

# E o que sobrar nas duas ficará como: 'Não Informado'
df_burnout_filtrado['uf_acidente'] = df_burnout_filtrado['uf_acidente'].fillna('Não Informado')

df_burnout_filtrado['uf_empregador'] = df_burnout_filtrado['uf_empregador'].fillna('Não Informado')

In [ ]:
### Tratamento das datas 'data_emissao_CAT' usará 'data_acidente' como um fallback

df_burnout_filtrado['data_emissao_cat'] = df_burnout_filtrado['data_emissao_cat'].fillna(df_burnout_filtrado['data_acidente'])

# data_afastamento: Mantém NaT para registros onde não houve afastamento formalizado


In [ ]:
### Validando os resultados

print('#### finalização dos Nulos ####')

print(
    f"Nulos em uf_acidente: {df_burnout_filtrado['uf_acidente'].isna().sum()} |"
    "Registros 'Não Informado'':"
    f"{(df_burnout_filtrado['uf_acidente'] == 'Não Informado').sum()}"
)

print('Nulos em uf_empregador:'
  f"{df_burnout_filtrado['uf_empregador'].isna().sum()} | Registros 'Não"
  "Informado':"
  f"{(df_burnout_filtrado['uf_empregador'] == 'Não Informado').sum()}"
)

print('Nulos em data_emissao_cat:'
    f"{df_burnout_filtrado['data_emissao_cat'].isna().sum()}"
)

print(
    'Nulos em data_afastamento (NaT mantidos com'
    f" sucesso): {df_burnout_filtrado['data_afastamento'].isna().sum()}"
)


#### finalização dos Nulos ####
Nulos em uf_acidente: 0 |Registros 'Não Informado'':48
Nulos em uf_empregador:0 | Registros 'NãoInformado':48
Nulos em data_emissao_cat:0
Nulos em data_afastamento (NaT mantidos com sucesso): 3353


In [ ]:
df_burnout_filtrado.info()

<class 'pandas.core.frame.DataFrame'>
Index: 18706 entries, 202 to 1739632
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   agente_causador       18706 non-null  object        
 1   data_acidente         18706 non-null  datetime64[ns]
 2   cbo_codigo            18706 non-null  int64         
 3   cbo_descricao         18706 non-null  object        
 4   cid_codigo            18706 non-null  object        
 5   cid_descricao         18706 non-null  object        
 6   cnae_codigo           18706 non-null  int64         
 7   cnae_descricao        18706 non-null  object        
 8   emitente_cat          18706 non-null  object        
 9   indica_obito          18706 non-null  object        
 10  municipio_empregador  18706 non-null  object        
 11  natureza_lesao        18706 non-null  object        
 12  origem_cadastramento  18706 non-null  object        
 13  parte_corpo_ating

Antes de finalizar a etapa de limpeza, vamos olhar mais minuciosamente para a coluna de agente_causador para entender se podemos melhora-la:

Visualizando a estrutura ela possui vários agentes causador, mas temos "ataque a ser vivo" e "ser vivo" que suponho ser a mesma coisa, na tabela do INSS, "Ser Vivo" se refere ao ser humano. Quando o médico/perito cadastra um caso de assédio moral, agressão verbal, ameaças, violência no trabalho ou assédio sexual exercido por chefia, clientes ou pacientes (ex: médicos/enfermeiros agredidos), o sistema enquadra formalmente em código de "Ataque de Ser Vivo / Agente Vivo"! Além também de ter o famoso {ñ class}, tem outras categorias estranhas também como: Edificio - Edificio / animal vivo...

Exposicao a Pressao e Pressao ambiente também acredito serem a mesma coisa, possui virgula nos finais.

A ideia aqui é entender em qual categoria os principais agentes se encaixam ou possívelmente estão duplicados e agrupa-los em no máximo 6 categorias para que possíveis gráficos não fiquem poluidos com inúmeros ruidos de agentes causadores.

In [ ]:
df_burnout_filtrado['agente_causador'].value_counts().head(20)

,count
agente_causador,
"Ataque de Ser Vivo,",4880
{ñ class},4149
"Ser Vivo, Nic",2813
"Esforco Excessivo, N",2074
"Agente do Acidente,",1335
Agente do Acidente I,803
Area ou Ambiente de,468
Exposicao a Pressao,224
"Pressao Ambiente, Ex",211


In [ ]:
#Verificando como realmente estão os nomes para fazer o mapeamento

for agente in df_burnout_filtrado['agente_causador'].unique():
  print(f"'{agente}'")

'Agente do Acidente I'
'{ñ class}'
'Ataque de Ser Vivo,'
'Agente do Acidente,'
'Ser Vivo, Nic'
'Esforco Excessivo, N'
'Veiculo, Nic'
'Rua e Estrada - Supe'
'Area ou Ambiente de'
'Rampa - Superficie U'
'Atrito ou Abrasao, N'
'Veiculo Rodoviario M'
'Impacto Sofrido por'
'Piso de Veiculo - Su'
'Reacao do Corpo a Mo'
'Pressao Ambiente, Ex'
'Exposicao a Pressao'
'Aeronave'
'Piso de Edificio - S'
'Edificio - Edificio'
'Motocicleta, Motonet'
'Esforco Excessivo ao'
'Chao - Superficie Ut'
'Superficie de Susten'
'Maquina de Costurar'
'Animal Vivo'
'Queda de Pes. em Mes'
'Energia'
'Vidraria, Fibra de V'
'Transportador com Fo'
'Arquivo, Fichario, E'
'Superficie e Estrutu'
'Escada Permanente Cu'
'Veiculo Sobre Trilho'
'Mesa, Carteira, Exce'
'Pressao Ambiente Alt'
'Escavacao (Para Edif'
'Edificio ou Estrutur'
'Atrito ou Abrasao po'
'Poco, Entrada, Galer'
'Ferramenta Manual se'
'Embalagem e Recipien'
'Maquina, Nic'
'Calcada ou Caminho p'
'Bicicleta'
'Passarela ou Platafo'
'Inalacao, Ingestao o'
'Ruid

UAU, toda essa quantidade absurda de categorias para Burnout acaba gerando muito ruido para a base de dados e para o modelo de ML futuramente, essa coluna precisa ser tratada urgente

A estrátegia aqui é a de `Agrupamento Macro` por Palavra-Chave (Regex).
Vamos criar um agrupamento funcional baseado nos pilares mais frequentes e relevantes para o Burnout, jogando o ruído de acidentes físicos pontuais em categorias genéricas limpas.

1. `Assedio_Agressao`: Tudo que contiver 'Ser Vivo', 'Ataque', 'Animal', 'Impacto de Pes'.

2. `Sobrecarga_Trabalho`: Termos com 'Esforco', 'Esforço', 'Reacao do Corpo', 'Atrito'.

3. `Pressao_Ambiental`: Termos com 'Pressao', 'Pressão', 'Ruido', 'Ruído', 'Temperatura', 'Poluicao'.

4. `Ambiente_Fisico`: Tudo sobre estrutura, como 'Edificio', 'Edifício', 'Chao', 'Piso', 'Escada', 'Area', 'Rua', 'Cadeira', 'Mobiliario', 'Mesa'.

5. `Maquinas_Ferramentas_Veiculos`: Qualquer máquina, ferramenta, veículo ou equipamento físico (ex: 'Veiculo', 'Maquina', 'Ferramenta', 'Eletrico', 'Trator', 'Bicicleta').

6. `Nao_Informado`: Preenchimentos de sujeira do banco como 'class', 'nan', 'None'.

7. `Outros`: Qualquer outro caso físico raro que sobrou da lista longa.

In [ ]:
#### Conversão para string e remoção de espaços extras se houver
agentes = df_burnout_filtrado['agente_causador'].astype(str).str.strip()

## Definição das regras de busca por padrão de texto

condicoes = [
    # 1. Fator Humano / Assédio / Agressão
    agentes.str.contains(
        'Ser Vivo|Ataque|Animal|Impacto de Pes', case=False, na=False
    ),
    # 2. Sobrecarga e Esforço
    agentes.str.contains(
        'Esforco|Esforço|Reacao do Corpo|Atrito', case=False, na=False
    ),
    # 3. Pressão, Estresse Ambiental e Ruído
    agentes.str.contains(
        'Pressao|Pressão|Ruido|Ruído|Temperatura|Poluicao|Poluição',
        case=False,
        na=False,
    ),
    # 4. Estrutura Física / Instalações de Escritório / Terreno
    agentes.str.contains(
        'Edificio|Edifício|Chao|Chão|Piso|Escada|Area|Área|Rua|Calcada|Calcada|Cadeira|Mobiliario|Mobiliário|Mesa|Balcao|Balcão|Platafo|Ponte|Estrutur',
        case=False,
        na=False,
    ),
    # 5. Máquinas, Equipamentos, Ferramentas e Veículos (Agrupando acidentes físicos)
    agentes.str.contains(
        'Veiculo|Veículo|Maquina|Máquina|Ferramenta|Eletrico|Elétrico|Equip|Trator|Bicicleta|Aeronave|Empilhadeira|Guindaste|Motor|Bomba|Prensa|Forno|Vaso|Tijolo|Faca|Madeira|Fogo|Gás|Gas|Produto|Substancia|Substância|Vidraria|Ceramica|Cerâmica',
        case=False,
        na=False,
    ),
    # 6. Sujeiras do sistema / Nulos
    agentes.str.contains('class|nan|None', case=False, na=False),
]

### Rotulo atualizados e padronizados para ML e Dashboard

categorias_macro = [
    'Assedio_Agressao',
    'Sobrecarga_trabalho',
    'Pressao_ambiental',
    'Ambiente_fisico',
    'Maquinas_equipamentos_veiculos',
    'Nao_Informado',
]


### Aplicando o mapeamento condicional
df_burnout_filtrado['agente_causador'] = np.select(condicoes, categorias_macro, default='Outros')


### Validando o Resultado

print('#### Distribuição das categorias macro #####')
print(df_burnout_filtrado['agente_causador'].value_counts())

#### Distribuição das categorias macro #####
agente_causador
Assedio_Agressao                  7782
Nao_Informado                     4149
Outros                            2501
Sobrecarga_trabalho               2229
Ambiente_fisico                    989
Maquinas_equipamentos_veiculos     603
Pressao_ambiental                  453
Name: count, dtype: int64


#### Agora possuimos algo mais macro, mais centralizado, enxuto e sem ruidos.

Reduzimos centenas de strings quebradas para poucas categorias sintéticas, limpas e com alta densidade estatística.

Preservamos o sinal mais forte da base (os casos de Agressão/Assédio e Sobrecarga).

###Agora faremos o mesmo para as colunas cid


In [ ]:

df_burnout_filtrado['cid_descricao'].value_counts()

,count
cid_descricao,
Z73.0 Esgotamento,5936
F41.1 Ansiedade Gene,2772
F41.2 Transt Misto A,1605
F43.1 Estado de Stre,1322
F43.0 Reacao Aguda a,1121
{ñ class},1055
F32.2 Episodio Depre,943
F41.0 Transt de Pani,728
F43.2 Transt de Adap,534


In [ ]:
df_burnout_filtrado['cid_codigo'].value_counts()

,count
cid_codigo,
Z730,5936
F411,2772
F412,1605
F431,1322
F430,1121
F322,943
F410,728
F432,534
F41,417


As colunas estão claramente precisando de uma limpeza pois se trata de erro de sistema quando os dados foram preenchidos.

E pensando bem as duas colunas acabam trazendo a mesma informaçao que é o código da doença

- Z730 / Z73.0 Esgotamento: É a maior ocorrência disparada (5.936 registros). Na classificação da OMS, Z73.0 é a CID oficial do Burnout (Esgotamento / Síndrome de Burnout).

- Grupo F (Transtornos Mentais e Comportamentais): Os demais códigos são comorbidades diretas do diagnóstico ou diagnósticos correlatos (F41 = Ansiedade, F32/F33 = Depressão, F43 = Reações ao Estresse).

- Estratégia que iremos adotar:

`Limpar a coluna cid_codigo`: Remover qualquer ocorrência de NU, NUL ou espaços extras no final do código usando Expressões Regulares (regex).

`Formatá-lo no padrão da CID-10`: Inserir o ponto antes do último dígito se houver 4 caracteres (ex: Z730 > Z73.0).

`Mapear a Descrição Oficial Limpa`: Em vez de depender do texto cortado do INSS (cid_descricao), podemos usar um dicionário exato com a descrição oficial completa de cada CID.

In [ ]:
### Limpeza dos sufixos ruins da base ( NUL, NU, {'ñ class'}) em cid_código

df_burnout_filtrado['cid_codigo_limpo'] = (
    df_burnout_filtrado['cid_codigo']
    .astype(str)
    .str.strip()
    .str.replace(r'NUL$|NU$', '', regex=True) # Removendo os NU do final
    .str.replace(
        r'{\ñ class}|nan|None', 'Não Informado', regex=True
    )
)


## Formatando os códigos para o padrão CID-10
def formatar_cid(codigo):
  if codigo == 'Não Informado' or len(codigo) < 3:
    return codigo
  if len(codigo) == 4 and '.' not in codigo:
    return f'{codigo[:3]}.{codigo[3]}'
  return codigo


df_burnout_filtrado['cid_codigo_limpo'] = df_burnout_filtrado['cid_codigo_limpo'].apply(formatar_cid)

#Dicionário das descrições oficiais
descricoes_cid = {
    'Z73.0': 'Esgotamento (Síndrome de Burnout)',
    'F41.1': 'Ansiedade Generalizada',
    'F41.2': 'Transtorno Misto Ansioso e Depressivo',
    'F43.1': 'Estado de Estresse Pós-Traumático',
    'F43.0': 'Reação Aguda ao Estresse',
    'F32.2': 'Episódio Depressivo Grave sem Sintomas Psicóticos',
    'F41.0': 'Transtorno de Pânico (Ansiedade Paroxística Episódica)',
    'F43.2': 'Transtornos de Adaptação',
    'F41.9': 'Transtorno Ansioso Não Especificado',
    'F32.1': 'Episódio Depressivo Moderado',
    'F33.2': 'Transtorno Depressivo Recorrente (Grave)',
    'F32.0': 'Episódio Depressivo Leve',
    'F33.1': 'Transtorno Depressivo Recorrente (Moderado)',
    'F33.0': 'Transtorno Depressivo Recorrente (Leve)',
    'F43.8': 'Outras Reações ao Estresse Grave',
    'F43.9': 'Reação ao Estresse Grave Não Especificada',
}

## Mapeando a descrição completa ou preencher com 'Outros''
df_burnout_filtrado['cid_descricao_limpa'] = (
    df_burnout_filtrado['cid_codigo_limpo']
    .map(descricoes_cid)
    .fillna('Outros Transtornos')
)


## Substituir nas colunas principais e remover auxiliares
df_burnout_filtrado['cid_codigo'] = df_burnout_filtrado['cid_codigo_limpo']
df_burnout_filtrado['cid_descricao'] = df_burnout_filtrado['cid_descricao_limpa']

df_burnout_filtrado.drop(columns=['cid_codigo_limpo', 'cid_descricao_limpa'], inplace=True)


### Validando o Resultado

print("\n###### Cids padronizadas (Código) ######\n")
print(df_burnout_filtrado['cid_codigo'].value_counts())

print("\n###### Cids padronizadas (Descrição) ######\n")
print(df_burnout_filtrado['cid_descricao'].value_counts())





###### Cids padronizadas (Código) ######

cid_codigo
Z73.0    6175
F41.1    2869
F41.2    1660
F43.1    1364
F43.0    1159
F32.2     971
F41.0     764
F43.2     552
F41       446
F32.1     395
F43.3     356
F43       341
F33.2     299
F41.9     257
F32       189
F33.1     169
F33.3     143
F43.8     128
F32.3     118
F33        78
F43.9      63
F32.9      48
F32.0      46
F41.3      34
F41.8      30
F32.8      23
F33.0      15
F33.9       8
F33.8       5
F33.4       1
Name: count, dtype: int64

###### Cids padronizadas (Descrição) ######

cid_descricao
Esgotamento (Síndrome de Burnout)                         6175
Ansiedade Generalizada                                    2869
Outros Transtornos                                        1820
Transtorno Misto Ansioso e Depressivo                     1660
Estado de Estresse Pós-Traumático                         1364
Reação Aguda ao Estresse                                  1159
Episódio Depressivo Grave sem Sintomas Psicóticos          971

In [ ]:
df_burnout_filtrado.head()

,agente_causador,data_acidente,cbo_codigo,cbo_descricao,cid_codigo,cid_descricao,cnae_codigo,cnae_descricao,emitente_cat,indica_obito,...,parte_corpo_atingida,genero,tipo_acidente,uf_empregador,data_afastamento,data_nascimento,data_emissao_cat,tipo_empregador,idade,uf_acidente
202,Outros,2025-03-20,373140,{ñ class},F33.2,Transtorno Depressivo Recorrente (Grave),4742,Comercio Varejista d,Empregador,Não,...,Sistema Nervoso,Feminino,Doença,Pernambuco,2025-03-20,1998-08-12,2025-07-23,Cnpj/Cgc,27.0,PA
423,Nao_Informado,2025-07-22,519110,519110-Motociclista,F41.1,Ansiedade Generalizada,4530,Comercio de Pecas e,Empregador,Não,...,"Cabeca, Partes Multi",Masculino,Doença,São Paulo,2025-07-15,1992-12-12,2025-07-24,Cnpj/Cgc,33.0,MA
596,Assedio_Agressao,2025-05-28,141705,141705-Ger. Prod. Ba,Z73.0,Esgotamento (Síndrome de Burnout),6422,"Bancos Multiplos, co",Sindicato,Não,...,Sistema Nervoso,Feminino,Doença,São Paulo,2025-05-28,1976-01-08,2025-07-10,Cnpj/Cgc,49.0,MA
702,Nao_Informado,2025-06-12,232105,232105-Prof. Artes n,F41.2,Transtorno Misto Ansioso e Depressivo,8541,Educacao Profissiona,Segurado/Dependente,Não,...,Sistemas e Aparelhos,Feminino,Doença,Rio de Janeiro,2025-06-12,1982-11-11,2025-07-16,Cnpj/Cgc,43.0,TO
713,Outros,2025-05-02,252210,252210-Contador,F32.1,Episódio Depressivo Moderado,6110,Telecomunicacoes por,Sindicato,Não,...,Localizacao da Lesao,Feminino,Típico,São Paulo,2025-05-02,1978-06-28,2025-07-31,Cnpj/Cgc,47.0,MA


- Decidi olhar para a coluna empregador para verificar se temos todos os registros como CNPJ, se sim vamos deleta-la da base

In [ ]:
df_burnout_filtrado['tipo_empregador'].value_counts()

,count
tipo_empregador,
Cnpj/Cgc,16066
nan,2530
Ignorado,78
Cpf,18
{ñ class},12
Nit,2


Olhando a distribuição, temos:

`Cnpj/Cgc`: 16.066 registros (~85.9% da base)

`nan`: 2.530 registros (~13.5% da base)

`Ignorado, Cpf, {\ñ class}, Nit`: 110 registros no total (~0.6% da base)

A decisão de mante-la acredito não ser viável pois temos mais de 85% dela preenchida e pertencem a apenas uma categoria, resultando em baixissima variância.

Temos também outros motivos:

Sem poder preditivo: Como quase todo mundo na base trabalha para pessoa jurídica (CNPJ), o modelo não ganha nenhum poder de discriminação ou padrão útil ao olhar essa variável.

Complexidade desnecessária: Ela não ajuda a diferenciar se um trabalhador terá maior tempo de afastamento, determinado CID ou agente causador específico.

In [ ]:
## Deletando a coluna 'tipo_empregador'

if 'tipo_empregador' in df_burnout_filtrado.columns:
  df_burnout_filtrado.drop(columns=['tipo_empregador'], inplace=True)

print('Coluna "tipo_empregador" removida com sucesso' )

Coluna "tipo_empregador" removida com sucesso


- E decidi também olhar as colunas `cbo_codigo`	`cbo_descricao` porque pelo que vi no head acima temos informaçoes também redundantes nelas

In [ ]:
df_burnout_filtrado['cbo_codigo'].value_counts()

,count
cbo_codigo,
253215,2302
413210,1185
141710,941
415205,914
142105,591
...,...
111505,1
841456,1
262125,1


In [ ]:
df_burnout_filtrado['cbo_descricao'].value_counts()

,count
cbo_descricao,
253215-Gerente Conta,2302
{ñ class},1369
413210-Caixa Banco,1185
141710-Ger. Agência,941
415205-Carteiro,914
...,...
764210-Montador de C,1
715205-Calceteiro,1
712120-Oper. Britado,1


Para entender esses código precisamos entender como a CBO classifica o grande grupo ocupacional. A Hierarquia acontece no primeiro digito do código.

| Código (1º Dígito) | Grande Grupo Ocupacional da CBO | Exemplo de Ocupações |
| :---: | :--- | :--- |
| **0** | Forças Armadas, Policiais e Bombeiros | Policial Militar, Bombeiro |
| **1** | Dirigentes e Gerentes | Gerente de Agência, Diretor |
| **2** | Profissionais das Ciências e das Artes (Nível Superior) | Gerente de Conta, Analista, Médico |
| **3** | Técnicos de Nível Médio | Técnico de Enfermagem, Agente |
| **4** | Trabalhadores de Serviços Administrativos | Caixa de Banco, Carteiro, Auxiliar |
| **5** | Trabalhadores dos Serviços e Comércio | Vendedor, Atendente, Vigilante |
| **6** | Trabalhadores Agropecuários, Florestais e da Pesca | Trabalhador Rural, Tratorista |
| **7** | Trabalhadores da Produção Industrial (Parte I) | Montador, Pedreiro, Mecânico |
| **8** | Trabalhadores da Produção Industrial (Parte II) | Operador de Máquinas/Instalações |
| **9** | Trabalhadores de Manutenção e Reparação | Eletricista, Supervisor de Manutenção |

Agora com base nessa descrição mais detalhada vamos criar uma outra coluna fazendo uma especifição mais macro dessas profissões, para que o modelo não fique com problemas de alta dimensionalidade mais a frente.

In [ ]:
### Tratando o {ñ class} e as sujeiras em cbo_descricao
df_burnout_filtrado['cbo_descricao'] = (
    df_burnout_filtrado['cbo_descricao']
    .astype(str)
    .str.strip()
    .str.replace(
        r'{\ñ class}|{ñ class}|nan|None', 'Não Informado', regex = True
    )
)

### Separando o código da descricão (removendo os 6 codigos e hífem no texto)
df_burnout_filtrado['cbo_descricao_limpa'] = (
    df_burnout_filtrado['cbo_descricao']
    .str.replace(r'^\d{6}\s*-\s*', '', regex=True)
    .str.strip()
)


## Limpeza do código

df_burnout_filtrado['cbo_codigo_limpo'] = (
    df_burnout_filtrado['cbo_codigo']
    .astype(str)
    .str.extract(r'(\d{6})')[0]  # Extrai apenas os 6 dígitos numéricos
    .fillna('Não Informado')
)


# 4. Criando a coluna 'cbo_grupo_macro' baseada no PRIMEIRO DÍGITO do CBO (Engenharia de Features para ML)
mapeamento_grupos_cbo = {
    '1': 'Gestores_Diretores_Gerentes',
    '2': 'Profissionais_Especialistas_Nivel_Superior',
    '3': 'Tecnicos_Nivel_Medio',
    '4': 'Servicos_Administrativos_Atendimento',
    '5': 'Comercio_e_Servicos',
    '6': 'Agropecuaria_Forestal_Pesca',
    '7': 'Producao_Industrial_Artesanato',
    '8': 'Operadores_Instalacoes_Maquinas',
    '9': 'Manutencao_E_Reparacao',
}


# Extraindo o primeiro dígito
primeiro_digito = df_burnout_filtrado['cbo_codigo_limpo'].str[0]

# Mapeando para os grandes grupos ocupacionais
df_burnout_filtrado['cbo_grupo_macro'] = (
    primeiro_digito.map(mapeamento_grupos_cbo).fillna('Não Informado')
)

# Substituindo nas colunas finais e limpar auxiliares
df_burnout_filtrado['cbo_codigo'] = df_burnout_filtrado['cbo_codigo_limpo']
df_burnout_filtrado['cbo_descricao'] = df_burnout_filtrado[
    'cbo_descricao_limpa'
]
df_burnout_filtrado.drop(
    columns=['cbo_codigo_limpo', 'cbo_descricao_limpa'], inplace=True
)

# Validando e visualizando o resultado do agrupamento macro
print('#### Distribuiçao do grupos ocupacionais (CBO) ####')
print(df_burnout_filtrado['cbo_grupo_macro'].value_counts())

#### Distribuiçao do grupos ocupacionais (CBO) ####
cbo_grupo_macro
Profissionais_Especialistas_Nivel_Superior    5208
Servicos_Administrativos_Atendimento          4709
Gestores_Diretores_Gerentes                   3264
Tecnicos_Nivel_Medio                          2096
Comercio_e_Servicos                           2040
Producao_Industrial_Artesanato                1091
Operadores_Instalacoes_Maquinas                176
Manutencao_E_Reparacao                          86
Agropecuaria_Forestal_Pesca                     34
Não Informado                                    2
Name: count, dtype: int64


In [ ]:
df_burnout_filtrado.head()

,agente_causador,data_acidente,cbo_codigo,cbo_descricao,cid_codigo,cid_descricao,cnae_codigo,cnae_descricao,emitente_cat,indica_obito,...,parte_corpo_atingida,genero,tipo_acidente,uf_empregador,data_afastamento,data_nascimento,data_emissao_cat,idade,uf_acidente,cbo_grupo_macro
202,Outros,2025-03-20,373140,Não Informado,F33.2,Transtorno Depressivo Recorrente (Grave),4742,Comercio Varejista d,Empregador,Não,...,Sistema Nervoso,Feminino,Doença,Pernambuco,2025-03-20,1998-08-12,2025-07-23,27.0,PA,Tecnicos_Nivel_Medio
423,Nao_Informado,2025-07-22,519110,Motociclista,F41.1,Ansiedade Generalizada,4530,Comercio de Pecas e,Empregador,Não,...,"Cabeca, Partes Multi",Masculino,Doença,São Paulo,2025-07-15,1992-12-12,2025-07-24,33.0,MA,Comercio_e_Servicos
596,Assedio_Agressao,2025-05-28,141705,Ger. Prod. Ba,Z73.0,Esgotamento (Síndrome de Burnout),6422,"Bancos Multiplos, co",Sindicato,Não,...,Sistema Nervoso,Feminino,Doença,São Paulo,2025-05-28,1976-01-08,2025-07-10,49.0,MA,Gestores_Diretores_Gerentes
702,Nao_Informado,2025-06-12,232105,Prof. Artes n,F41.2,Transtorno Misto Ansioso e Depressivo,8541,Educacao Profissiona,Segurado/Dependente,Não,...,Sistemas e Aparelhos,Feminino,Doença,Rio de Janeiro,2025-06-12,1982-11-11,2025-07-16,43.0,TO,Profissionais_Especialistas_Nivel_Superior
713,Outros,2025-05-02,252210,Contador,F32.1,Episódio Depressivo Moderado,6110,Telecomunicacoes por,Sindicato,Não,...,Localizacao da Lesao,Feminino,Típico,São Paulo,2025-05-02,1978-06-28,2025-07-31,47.0,MA,Profissionais_Especialistas_Nivel_Superior


- Outra coluna que com certeza merece a nossa atençao é a cnae_descricao (Classificação Nacional de Atividades Econônomicas ). Ela irá nos dizer qual setor econômico o Burnout está mais concentrado

In [ ]:

df_burnout_filtrado['cnae_descricao'].value_counts()

,count
cnae_descricao,
"Bancos Multiplos, co",5098
Bancos Comerciais,1628
Atividades de Correi,1067
Atividades de Atendi,896
Comercio Varejista d,844
...,...
Torrefacao e Moagem,1
Avaliacao de Riscos,1
Transportes Aquaviar,1


De Cara o setor Financeiro/ Bancário já aparece liderando a posiçao de setor mais afetado. Porém a quantidade de categorias são muitas, entao usando a mesma estratégia anterior vamos consolidar apenas em umas 7 categorias, e pra isso podemos usar o código do cnae.

Devemos tratar essa coluna pelos seguintes fatos:

`Descrições truncadas`: Nomes cortados pela metade (ex: Bancos Multiplos, co, Atividades de Correi).

`Cardinalidade alta (314 categorias):` Deixar 314 setores para o modelo de Machine Learning gera muita dispersão.

In [ ]:
# Limpeza inicial de espaços e caracteres estranhos
df_burnout_filtrado['cnae_descricao_limpa'] = (
    df_burnout_filtrado['cnae_descricao']
    .astype(str)
    .str.strip()
    .str.replace(
        r'{\ñ class}|{ñ class}|nan|None', 'Não Informado', regex=True
    )
)

# Definição das Regras Regex baseadas nos fragmentos da coluna
condicoes_cnae = [
    # 1. Serviços Financeiros, Bancos e Seguros (Maioria dabase)
    df_burnout_filtrado['cnae_descricao_limpa'].str.contains(
        r'Banco|Financ|Credito|Seguro|Caixas Econ|Previdencia|Securitizacao|Sociedades de C|Agencias de Fom|Holdings|Gestaao de Ar',
        case=False,
        regex=True,
    ),
    # 2. Correios, Logística e Transporte
    df_burnout_filtrado['cnae_descricao_limpa'].str.contains(
        r'Correi|Transporte|Terminais|Gestao de Portos|Carga e Descarga|Navegacao|Armazenamento',
        case=False,
        regex=True,
    ),
    # 3. Teleatendimento, Call Center e Apoio
    df_burnout_filtrado['cnae_descricao_limpa'].str.contains(
        r'Teleat|Atendi|Atividades de Apoio|Atividades de Servic|Servicos Combinados|Suporte Tecnico',
        case=False,
        regex=True,
    ),
    # 4. Saúde e Serviços Sociais
    df_burnout_filtrado['cnae_descricao_limpa'].str.contains(
        r'Atenca|Saude|Hospital|Medico|Veterinar|Planos de Saude',
        case=False,
        regex=True,
    ),
    # 5. Comércio (Varejo e Atacado)
    df_burnout_filtrado['cnae_descricao_limpa'].str.contains(
        r'Comercio|Varej|Atacad|Concessionarias', case=False, regex=True
    ),
    # 6. Administração Pública, Defesa e Segurança
    df_burnout_filtrado['cnae_descricao_limpa'].str.contains(
        r'Administracao Public|Defesa|Justica|Seguranca e Ordem|Seguridade Social',
        case=False,
        regex=True,
    ),
    # 7. Educação
    df_burnout_filtrado['cnae_descricao_limpa'].str.contains(
        r'Educacao|Ensino', case=False, regex=True
    ),
    # 8. Indústria, Fabricação, Construção e Extração (Pega as dezenas de 'Fabricacao de...')
    df_burnout_filtrado['cnae_descricao_limpa'].str.contains(
        r'Fabricacao|Producao|Construcao|Obras|Metalurgia|Fundicao|Moagem|Torrefacao|Extracao|Cultivo|Criacao|Abate|Confeccao|Eletrica|Geracao de Energ|Distribuicao de Ener|Captacao, Tratamento|Tratamento e Disposi',
        case=False,
        regex=True,
    ),
    # 9. Dados não informados
    df_burnout_filtrado['cnae_descricao_limpa'].str.contains(
        r'Não Informado', case=False, regex=True
    ),
]

setores_macro = [
    'Servicos_Financeiros_E_Bancos',
    'Correios_Logistica_E_Transporte',
    'Teleatendimento_E_Servicos_Apoio',
    'Saude_E_Servicos_Sociais',
    'Comercio_Varejo_E_Atacado',
    'Administracao_Publica_E_Defesa',
    'Educacao',
    'Industria_Agro_E_Construcao',
    'Nao_Informado',
]

# Aplicando o mapeamento
df_burnout_filtrado['cnae_setor_macro'] = np.select(
    condicoes_cnae, setores_macro, default='Outros_Servicos_E_Tecnologia'
)

# Validando os dos resultados
print('####### Destribuiçao setor tratada ########')
print(df_burnout_filtrado['cnae_setor_macro'].value_counts())

####### Destribuiçao setor tratada ########
cnae_setor_macro
Servicos_Financeiros_E_Bancos       7301
Outros_Servicos_E_Tecnologia        3022
Teleatendimento_E_Servicos_Apoio    1979
Correios_Logistica_E_Transporte     1878
Comercio_Varejo_E_Atacado           1422
Industria_Agro_E_Construcao         1284
Administracao_Publica_E_Defesa       773
Educacao                             548
Saude_E_Servicos_Sociais             433
Nao_Informado                         66
Name: count, dtype: int64


In [ ]:
df_burnout_filtrado.head()

,agente_causador,data_acidente,cbo_codigo,cbo_descricao,cid_codigo,cid_descricao,cnae_codigo,cnae_descricao,emitente_cat,indica_obito,...,tipo_acidente,uf_empregador,data_afastamento,data_nascimento,data_emissao_cat,idade,uf_acidente,cbo_grupo_macro,cnae_descricao_limpa,cnae_setor_macro
202,Outros,2025-03-20,373140,Não Informado,F33.2,Transtorno Depressivo Recorrente (Grave),4742,Comercio Varejista d,Empregador,Não,...,Doença,Pernambuco,2025-03-20,1998-08-12,2025-07-23,27.0,PA,Tecnicos_Nivel_Medio,Comercio Varejista d,Comercio_Varejo_E_Atacado
423,Nao_Informado,2025-07-22,519110,Motociclista,F41.1,Ansiedade Generalizada,4530,Comercio de Pecas e,Empregador,Não,...,Doença,São Paulo,2025-07-15,1992-12-12,2025-07-24,33.0,MA,Comercio_e_Servicos,Comercio de Pecas e,Comercio_Varejo_E_Atacado
596,Assedio_Agressao,2025-05-28,141705,Ger. Prod. Ba,Z73.0,Esgotamento (Síndrome de Burnout),6422,"Bancos Multiplos, co",Sindicato,Não,...,Doença,São Paulo,2025-05-28,1976-01-08,2025-07-10,49.0,MA,Gestores_Diretores_Gerentes,"Bancos Multiplos, co",Servicos_Financeiros_E_Bancos
702,Nao_Informado,2025-06-12,232105,Prof. Artes n,F41.2,Transtorno Misto Ansioso e Depressivo,8541,Educacao Profissiona,Segurado/Dependente,Não,...,Doença,Rio de Janeiro,2025-06-12,1982-11-11,2025-07-16,43.0,TO,Profissionais_Especialistas_Nivel_Superior,Educacao Profissiona,Educacao
713,Outros,2025-05-02,252210,Contador,F32.1,Episódio Depressivo Moderado,6110,Telecomunicacoes por,Sindicato,Não,...,Típico,São Paulo,2025-05-02,1978-06-28,2025-07-31,47.0,MA,Profissionais_Especialistas_Nivel_Superior,Telecomunicacoes por,Outros_Servicos_E_Tecnologia


- Com a consolidaçao da coluna Setor Macro, podemos deletar a coluna auxiliar e a coluna original, cnae_descricao, já que vamos utilizar a macro para fazer as análises e utilizar no modelo de ML

In [ ]:
### Removendo as colunas redundantes  de CNAE

colunas_remove = ['cnae_descricao', 'cnae_descricao_limpa']

# Apaga as colunas que realmente existem no dataframe

df_burnout_filtrado.drop(columns=[
    col
    for col in colunas_remove
    if col in df_burnout_filtrado.columns], inplace = True,
)

print('Limpeza concluida e colunas Cnae redundantes removidas')

Limpeza concluida e colunas Cnae redundantes removidas


In [ ]:
df_burnout_filtrado.head()

,agente_causador,data_acidente,cbo_codigo,cbo_descricao,cid_codigo,cid_descricao,cnae_codigo,emitente_cat,indica_obito,municipio_empregador,...,genero,tipo_acidente,uf_empregador,data_afastamento,data_nascimento,data_emissao_cat,idade,uf_acidente,cbo_grupo_macro,cnae_setor_macro
202,Outros,2025-03-20,373140,Não Informado,F33.2,Transtorno Depressivo Recorrente (Grave),4742,Empregador,Não,Petrolina,...,Feminino,Doença,Pernambuco,2025-03-20,1998-08-12,2025-07-23,27.0,PA,Tecnicos_Nivel_Medio,Comercio_Varejo_E_Atacado
423,Nao_Informado,2025-07-22,519110,Motociclista,F41.1,Ansiedade Generalizada,4530,Empregador,Não,São Paulo,...,Masculino,Doença,São Paulo,2025-07-15,1992-12-12,2025-07-24,33.0,MA,Comercio_e_Servicos,Comercio_Varejo_E_Atacado
596,Assedio_Agressao,2025-05-28,141705,Ger. Prod. Ba,Z73.0,Esgotamento (Síndrome de Burnout),6422,Sindicato,Não,São Paulo,...,Feminino,Doença,São Paulo,2025-05-28,1976-01-08,2025-07-10,49.0,MA,Gestores_Diretores_Gerentes,Servicos_Financeiros_E_Bancos
702,Nao_Informado,2025-06-12,232105,Prof. Artes n,F41.2,Transtorno Misto Ansioso e Depressivo,8541,Segurado/Dependente,Não,Rio de Janeir,...,Feminino,Doença,Rio de Janeiro,2025-06-12,1982-11-11,2025-07-16,43.0,TO,Profissionais_Especialistas_Nivel_Superior,Educacao
713,Outros,2025-05-02,252210,Contador,F32.1,Episódio Depressivo Moderado,6110,Sindicato,Não,São Paulo,...,Feminino,Típico,São Paulo,2025-05-02,1978-06-28,2025-07-31,47.0,MA,Profissionais_Especialistas_Nivel_Superior,Outros_Servicos_E_Tecnologia


### Última checagem para consolidar e salvar a base tratada,limpa e enxuta tanto para as análises exploratórias quanto para o Modelo de ML e a solução visual final.

In [ ]:
## Checagem rápida

print(
    f"Dimensões finais do dataset: {df_burnout_filtrado.shape[0]} linhas x {df_burnout_filtrado.shape[1]}"
)

print("\n########## Colunas presentes na base tratada ###########\n")
print(df_burnout_filtrado.columns.tolist())

##Salvando em PARQUET pois ele preserva a tipagem e é mais leve
df_burnout_filtrado.to_parquet('df_burnout_tratado.parquet', index=False)


## Salvando também uma cópia em CSV
df_burnout_filtrado.to_csv('df_burnout_filtrado_limpo.csv', index=False)

print("\n Base higienizada e salva com sucesso em 'df_burnout_tratado.parquet'!")

Dimensões finais do dataset: 18706 linhas x 23

########## Colunas presentes na base tratada ###########

['agente_causador', 'data_acidente', 'cbo_codigo', 'cbo_descricao', 'cid_codigo', 'cid_descricao', 'cnae_codigo', 'emitente_cat', 'indica_obito', 'municipio_empregador', 'natureza_lesao', 'origem_cadastramento', 'parte_corpo_atingida', 'genero', 'tipo_acidente', 'uf_empregador', 'data_afastamento', 'data_nascimento', 'data_emissao_cat', 'idade', 'uf_acidente', 'cbo_grupo_macro', 'cnae_setor_macro']

 Base higienizada e salva com sucesso em 'df_burnout_tratado.parquet'!


# Resumo Técnico: Pipeline de Higienização e Preparação de Dados (CAT / INSS - Burnout)

###Projeto: Análise e Modelagem Preditiva de Afastamentos por Burnout / Transtornos Mentais

- Etapa: Engenharia e Qualidade de Dados
- Status: Concluído / Base Consolidada.

## Visão Geral e Objetivos
Este resumo detalha as decisões técnicas, diagnósticos de qualidade e estratégias de transformação aplicadas sobre a base de dados bruta de Comunicação de Acidente de Trabalho (CAT/INSS).

O objetivo principal desta etapa foi transformar uma base legada, ruidosa e redundante em um dataset de alta integridade, otimizado tanto para a construção de **dashboards analíticos em etapas posteriores**, quanto para o treinamento de modelos de **Machine Learning**.

#### 2. Diagnóstico de Sujeira e Problemas Encontrados

Durante a fase de exploração e auditoria dos dados, identificaram-se gargalos críticos de qualidade:

  - `Sufixos e Raciocínio de Banco Legado (NU / NUL / {ñ class}`: Preenchimentos automáticos incorretos de formulários do INSS gerando códigos de CID corrompidos (ex: Z730NU em vez de Z730).Rótulos de inconsistência ({ñ class}, nan, Ignorado) espalhados por variáveis categóricas essenciais.

  - `Redundância Informativa`: Colunas duplicando dados (ex: código embutido no texto da descrição).

  - `Strings Truncadas`: Campos como a descrição da CNAE e CBO cortados em 20–22 caracteres no banco de origem.

  - `Alta Cardinalidade`: Variáveis como CBO (753 categorias) e CNAE (314 categorias) que inviabilizavam a codificação direta para Machine Learning sem gerar esparsidade excessiva.

  `Variáveis de Baixíssima Variância`: Campos com mais de 85% de concentração em uma única classe unânime (tipo_empregador), agregando ruído sem valor preditivo.

#### 3. Estratégias e Decisões Técnicas Tomadas

**3.1. Tratamento de CIDs (Classificação Internacional de Doenças)**

  - Padronização Regex: Removeram-se os sufixos NU/NUL e aplicou-se a formatação padrão CID-10 com ponto decimal (ex: Z730  Z73.0).

  - Enriquecimento por Dicionário Oficial: Mapearam-se as descrições oficiais completas da CID-10, substituindo os textos cortados do INSS.

**3.2. Engenharia de Recursos na CBO (Ocupações)Agrupamento por Grande Grupo (1º Dígito):**
  - Em vez de submeter 753 cargos ao modelo, utilizou-se a hierarquia oficial da CBO. Pelo primeiro dígito do código, criou-se a feature cbo_grupo_macro, reduzindo a cardinalidade para 10 categorias estratégicas (ex: Dirigentes/Gerentes, Profissionais das Ciências, Serviços Administrativos).

  - Desacoplamento de Texto: A coluna de descrição foi higienizada removendo a duplicidade do código numérico.

**3.3. Agrupamento Preditivo da CNAE (Setores Econômicos)**
  - Tratamento de Textos Truncados por Regex: Diante das 314 variações cortadas (ex: Bancos Multiplos, co), aplicou-se mapeamento baseado em expressões regulares nos radicais das palavras. Consolidação em Setores Macro: A variável foi colapsada em 9 macrosetores da economia (ex: Serviços Financeiros e Bancos, Correios e Logística, Saúde e Serviços Sociais, Indústria e Construção).

**3.4. Descarte de Colunas Irrelevantes**
 - Remoção de tipo_empregador: Devido à dominância esmagadora da categoria CNPJ/CGC (~86%), a coluna foi eliminada por falta de poder de discriminação estatística. Eliminação de Colunas Auxiliares/Sujas: Todas as colunas temporárias de limpeza e textos redundantes/cortados foram deletadas ao final do pipeline.


4. Resumo das principais mudanças da Base

| Coluna Original | Problema Identificado | Ação / Transformação Aplicada | Resultado Final |
| :--- | :--- | :--- | :--- |
| **`cid_codigo`** / **`cid_descricao`** | Sufixos `NU`/`NUL`, sem ponto decimal, texto truncado. | Limpeza via Regex, formatação CID-10 e mapeamento oficial completo. | Código e Descrição padronizados (`Z73.0` - *Esgotamento*). |
| **`cbo_codigo`** / **`cbo_descricao`** | Redundância e 753 cargos (alta cardinalidade). | Separação de código/texto + criação de `cbo_grupo_macro` via 1º dígito. | `cbo_descricao` limpa e `cbo_grupo_macro` (10 categorias). |
| **`cnae_descricao`** | 314 categorias com texto truncado a 20 caracteres. | Agrupamento lógico por padrão Regex nos radicais de texto. | `cnae_setor_macro` (Reduzido para 9 setores econômicos). |
| **`tipo_empregador`** | Baixíssima variância (>85% CNPJ) e 13% de nulos. | Descarte da coluna por irrelevância preditiva (*Feature Selection*). | Coluna removida do *dataset*. |



5. Armazenamento e Próximos Passos
Formato de Exportação: A base final foi persistida em formato .parquet (df_burnout_limpo_pronto.parquet). A escolha do Parquet garante compressão eficiente, velocidade no carregamento do Streamlit e preservação rigorosa da tipagem das colunas (schema).

## Próximos Passos:

Análise Exploratória de Dados (EDA): Identificação dos cruzamentos de maior risco (ex: Setor Financeiro x Cargos de Atendimento x Faixa Etária).

Modelagem Preditiva: Aplicação de Encoding (OneHotEncoder / TargetEncoder) nas variáveis macro para treinamento de algoritmos de classificação/regressão.

Visualização: Construção dos filtros e gráficos no Streamlit consumindo o arquivo Parquet limpo.

## Começo da Análise explorátória:

Disponível em: https://colab.research.google.com/drive/1oiJhxwWkDm9kYFicFhU5Oct3tCT7libL?usp=sharing